In [26]:
import pandas as pd
import torch
import random

# Load the sequence data
val_seq_df = pd.read_csv('./data/sample_submission.csv')  # Assuming you've saved the above to a file

# Sort by resid just in case
#val_seq_df = val_seq_df.sort_values('resid')

# Extract sequence
sequence = val_seq_df['resname'].tolist()  # e.g., ['G', 'G', ..., 'C']

In [27]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class Residue3DPredictor(nn.Module):
    def __init__(self, window_size=5, num_classes=4):
        super().__init__()
        self.window_size = window_size

        # First convolution layer (4 channels → 32 channels)
        self.initial_conv = nn.Conv1d(num_classes, 32, kernel_size=9, padding=4)

        # 10 convolutional layers (32 → 32), with kernel_size=9, padding=4
        self.middle_convs = nn.Sequential(
            *[nn.Sequential(
                nn.Conv1d(32, 32, kernel_size=9, padding=4),
                nn.BatchNorm1d(32),
                nn.ReLU(),
                nn.Dropout(0.1)
            ) for _ in range(10)]
        )

        # Final convolution (32 → 128)
        self.final_conv = nn.Conv1d(32, 128, kernel_size=9, padding=4)

        # Pool to fixed window size
        self.pool = nn.AdaptiveAvgPool1d(output_size=window_size)

        # Fully connected to 3D coordinates
        self.fc = nn.Linear(128 * window_size, 3)

    def forward(self, x):
        x = F.relu(self.initial_conv(x))     # [B, 32, W]
        x = self.middle_convs(x)             # [B, 32, W]
        x = F.relu(self.final_conv(x))       # [B, 128, W]
        x = self.pool(x)                     # [B, 128, window_size]
        x = x.view(x.size(0), -1)            # [B, 128 * window_size]
        return self.fc(x)


In [28]:
def prepare_sequence(seq, window_size=5):
    mapping = {'A': [1, 0, 0, 0], 'U': [0, 1, 0, 0], 'C': [0, 0, 1, 0], 'G': [0, 0, 0, 1]}
    pad = [0, 0, 0, 0]
    input_data = []

    for i in range(len(seq)):
        context = []
        for j in range(i - window_size // 2, i + window_size // 2 + 1):
            if 0 <= j < len(seq):
                base = seq[j]
                one_hot = mapping.get(base, random.choice(list(mapping.values())))
                context.append(one_hot)
            else:
                context.append(pad)
        input_data.append(torch.tensor(context).T.float())  # shape: [4, window_size]

    return input_data

window_size = 5
inputs = prepare_sequence(sequence, window_size=window_size)
X_val = torch.stack(inputs).to('cuda' if torch.cuda.is_available() else 'cpu')  # [N, 4, window_size]

In [29]:
import pandas as pd
import torch

# === Define your model class here ===
model_classes = {
    'model1': Residue3DPredictor,
    'model2': Residue3DPredictor,
    'model3': Residue3DPredictor,
    'model4': Residue3DPredictor,
    'model5': Residue3DPredictor
}

model_names = ['model1', 'model2', 'model3', 'model4', 'model5']  # your model keys
# === Inputs: validation data (X_val), metadata (val_seq_df), model paths ===
model_paths = [
    'RNA_model.pt',
    'RNA_model.pt',
    'RNA_model.pt',
    'RNA_model.pt',
    'RNA_model.pt'
]

# Initialize output DataFrame with metadata
merged_df = val_seq_df[['ID', 'resname', 'resid']].copy()

# Predict with each model
for i, (name, path) in enumerate(zip(model_names, model_paths), 1):
    print(f"Loading model {i}: {path}")
    
    model = Residue3DPredictor(window_size=window_size)
    model = model_classes[name](window_size=window_size)
    model.load_state_dict(torch.load(path, map_location='cpu'))  # Or 'cuda'
    model.eval()
    model.to(X_val.device)

    with torch.no_grad():
        Y_pred = model(X_val)  # shape: [N, 3]

    # Store prediction in DataFrame
    merged_df[f'x_{i}'] = Y_pred[:, 0].cpu().numpy()
    merged_df[f'y_{i}'] = Y_pred[:, 1].cpu().numpy()
    merged_df[f'z_{i}'] = Y_pred[:, 2].cpu().numpy()

# === Save result ===
merged_df.to_csv('sample_submission_predicted.csv', index=False)
print("Saved: RNA_predicted.csv")


Loading model 1: RNA_model.pt
Loading model 2: RNA_model.pt
Loading model 3: RNA_model.pt
Loading model 4: RNA_model.pt
Loading model 5: RNA_model.pt
Saved: RNA_predicted.csv
